In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch 
import torch.nn.functional as F
import torch.nn as nn



#### Implementing CNN Architectures

In [2]:
from functools import partial

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

##### Tackling Fashion Mnist with CNN
- a `28` x `28` image of greyscale channel

In [4]:
torch.manual_seed(42)
default_conv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model=nn.Sequential(
    default_conv2d(in_channels=1, out_channels=64, kernel_size=7),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    default_conv2d(in_channels=64, out_channels=128),
    nn.ReLU(),
    default_conv2d(in_channels=128, out_channels=128),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    default_conv2d(in_channels=128, out_channels=256),
    nn.ReLU(),
    default_conv2d(in_channels=256, out_channels=256),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(), 
    nn.Linear(in_features=2304, out_features=128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),    
).to(device=device)

In [5]:
#%pip install torchmetrics

### Using torchmetrics classes
- `metric.update(preds, target)`: Accumulates predictions and targets into internal state for every batch
- `metric.compute()` : Computes final metric from accumulated state at end of batch
- `metric.reset()`: Clears internal state for next epoch
- `metric.forward(preds, target)`: Does update + compute for that batch only

In [6]:
# import torchmetrics 

# def evaluate_model(model, dataloader, metric):
#     model.eval()
#     metric.reset()
#     with torch.no_grad():
#         for x_batch, y_batch in dataloader:
#             x_batch, y_batch = x_batch.to(device), y_batch.to(device) 
#             y_pred = model(x_batch)
#             metric.update(y_pred, y_batch)
#     return metric.compute()




# def train_model(model, loss_fn, optimizer, metric, train_loader, valid_loader, epochs):
#     history = {"train_loses": [], "train_metrics": [], "valid_metrics": []}
#     for epoch in range(epochs):
#         total_loss=0.0
#         model.train()
#         metric.reset()
#         for x_batch, y_batch in train_loader: 
#             x_batch, y_batch = x_batch.to(device), y_batch.to(device)
#             y_pred = model(x_batch)
#             loss = loss_fn(y_pred, y_batch)
#             total_loss += loss.item()
#             loss.backward()
#             optimizer.step()
#             optimizer.zero_grad()
#             metric.update(y_pred, y_batch)
#         history["train_loses"].append(total_loss/ len(train_loader))
#         history["train_metrics"].append(metric.compute().item())
#         history["valid_metrics"].append(evaluate_model(model=model, dataloader=valid_loader, metric=metric).item())
        
#         print( f"Epoch {epoch +1} / {epochs},"
#               f"train loss {history['train_loses'][-1]:.4f}, "
#               f"train metrics {history['train_metrics'][-1]:.4f}, "
#               f"valid metrics {history['valid_metrics'][-1]:.4f}"
#         )
#     return history
   
    
            
    

In [7]:
import torchvision
import torchvision.transforms.v2 as T

In [8]:
toTensor = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True)
])

train_and_valid_dataset = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, transform=toTensor, download=True
)
test_dataset = torchvision.datasets.FashionMNIST(root="datasets", train=False, transform=toTensor, download=True)
torch.manual_seed(42)

train_data , valid_data = torch.utils.data.random_split(
    dataset=train_and_valid_dataset,
    lengths=[55_000, 5_000]
)

In [9]:
from torch.utils.data import DataLoader
torch.manual_seed(42)

train_loader = DataLoader(dataset=train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(dataset=valid_data, batch_size=32)
test_loader = DataLoader(dataset=test_dataset, batch_size=32)


In [10]:
# num_epochs= 25
# optimizer = torch.optim.AdamW(model.parameters())
# xentropy = nn.CrossEntropyLoss()
# accuracy = torchmetrics.Accuracy("multiclass", num_classes=10).to(device)

# history = train_model(
#                         model=model, 
#                         loss_fn=xentropy, 
#                         optimizer=optimizer, 
#                         metric=accuracy, 
#                         train_loader=train_loader,
#                         valid_loader=valid_loader,
#                         epochs=num_epochs,
#                         )



### LeNet-5 architecture implementation
- consist of conv blocks + pool blocks then finally fully connected block
- one novelity in this architecture is that Lecunn padded 28x28 image into 32x32 before feeding it into the network

In [12]:
### how to archieve the padding before feeding the image into the network
transforms = torchvision.transforms.Compose([
    torchvision.transforms.Pad(2),
    torchvision.transforms.ToTensor(), 
])

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.pool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16)
        self.pool2 = nn.AvgPool2d(kernel_size=2)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5)
        self.fc1 = nn.Linear(in_features=120, out_features=80),
        self.fc2 = nn.Linear(in_features=80, out_features=10)
        
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1) 

#### Implementation of Xception architure
- separable convolutional network